# BioJEPA v0.6 Data Prep - Notebook 2: Perturbation Embeddings

This notebook handles:
1. Loading all reference data (CRISPRi library, Adamson, Norman, UniProt)
2. Extracting unique perturbations per dataset
3. Mapping genes to protein sequences (simplified single mygene query)
4. Generating DNA, protein, and chemical embeddings
5. Building alignment pairs
6. Saving unified index mappings
7. Building skip sets for perturbations missing both sequence and target embeddings

**Inputs (from Notebook 1):**
- `gene_to_idx.json` - ENSG -> index for gene universe
- `gene_names.json` - index -> gene symbol
- `dataset_splits.json` - train/val/test perturbation sets per dataset

**Outputs:**
- `pert_embd/seq_banks/dna_embeddings.npy` - [N_dna, 1536]
- `pert_embd/seq_banks/dna_to_idx.json` - UNIFIED: sgID_AB, Adamson pert, Norman guide_id -> idx
- `pert_embd/seq_banks/chemical_embeddings.npy` - [N_chem, 1024]
- `pert_embd/seq_banks/chemical_to_idx.json` - UNIFIED: SMILES and drug names -> idx
- `pert_embd/target_banks/protein_targets.npy` - [N_genes, 320]
- `pert_embd/target_banks/gene_to_target_idx.json` - gene (ENSG or symbol) -> idx
- `pert_embd/input_to_id.json` - "GENE_mode_dataset" -> seq_idx (for pathway evals)
- `pert_embd/{train,val,test}/align_*.npz` - alignment pairs
- `skipped_perturbations.json` - perturbations to skip in training shards (missing both embeddings)

## Step 1: Setup & Load Reference Data

In [1]:
from pathlib import Path
from collections import defaultdict
from Bio import SeqIO, Entrez
from tqdm import tqdm
import pandas as pd
import numpy as np
import scanpy as sc
import mygene
import torch
import json
import gzip
import gc
import re
import pubchempy as pcp
import requests

Entrez.email = 'gptomics@gmail.com'

In [2]:
ref_dir = Path('/Users/djemec/data/jepa/reference_data')
data_dir = Path('/Users/djemec/data/jepa/v0_6')

seq_banks_dir = data_dir / 'pert_embd' / 'seq_banks'
target_banks_dir = data_dir / 'pert_embd' / 'target_banks'
pert_embd_dir = data_dir / 'pert_embd'
seq_banks_dir.mkdir(parents=True, exist_ok=True)
target_banks_dir.mkdir(parents=True, exist_ok=True)
for split in ['train', 'val', 'test']:
    (pert_embd_dir / split).mkdir(parents=True, exist_ok=True)

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(1337)
        device = 'cuda'
    print(f'using {device}')
    return device

device = get_device()

using cpu


In [3]:
def is_valid(val):
    if pd.isna(val):
        return False
    if isinstance(val, str) and (val.strip() == '' or val.strip().lower() == 'nan'):
        return False
    return True
    
def print_list_head(print_list, n=10):
    print({j:print_list[j] for j in list(print_list.keys())[:n]})

In [4]:
with open(data_dir / 'dataset_splits.json') as f:
    dataset_splits = json.load(f)

print(f'Loaded dataset splits for {len(dataset_splits)} datasets')

Loaded dataset splits for 6 datasets


In [5]:
crispr_df = pd.read_csv(ref_dir / 'crispr' / 'hcrispri_all.csv')
crispr_df['sgID_clean'] = crispr_df['sgID'].astype(str).str.replace(',', '-').str.strip()
sgid_to_seq = dict(zip(crispr_df['sgID_clean'], crispr_df['protospacer sequence'].str.strip()))
sgid_to_gene = dict(zip(crispr_df['sgID_clean'], crispr_df['gene'].str.strip()))
print(f'CRISPRi library: {len(sgid_to_seq)} sgID -> protospacer mappings')
print_list_head(sgid_to_seq)
print_list_head(sgid_to_gene)

CRISPRi library: 267846 sgID -> protospacer mappings
{'A1BG_-_58858617.23-P1': 'GGAGACCCAGCGCTAACCAG', 'A1BG_-_58858788.23-P1': 'GGGGCACCCAGGAGCGGTAG', 'A1BG_+_58858964.23-P1': 'GCTCCGGGCGACGTGGAGTG', 'A1BG_-_58858630.23-P1': 'GAACCAGGGGTGCCCAAGGG', 'A1BG_+_58858549.23-P1': 'GGCGAGGAACCGCCCAGCAA', 'A1BG_-_58858950.23-P1': 'GGCAGCGCAGGACGGCATCT', 'A1BG_-_58858915.23-P1': 'GAGCAGCTCGAAGGTGACGT', 'A1BG_-_58858991.23-P1': 'GTCCACGTCGCCCGGAGCTG', 'A1BG_-_58858562.23-P1': 'GCTGCAGGGCCTTTGCTGGG', 'A1BG_+_58858791.23-P1': 'GCCAGCACGCCGGCAACTAC'}
{'A1BG_-_58858617.23-P1': 'A1BG', 'A1BG_-_58858788.23-P1': 'A1BG', 'A1BG_+_58858964.23-P1': 'A1BG', 'A1BG_-_58858630.23-P1': 'A1BG', 'A1BG_+_58858549.23-P1': 'A1BG', 'A1BG_-_58858950.23-P1': 'A1BG', 'A1BG_-_58858915.23-P1': 'A1BG', 'A1BG_-_58858991.23-P1': 'A1BG', 'A1BG_-_58858562.23-P1': 'A1BG', 'A1BG_+_58858791.23-P1': 'A1BG'}


In [6]:
adamson_df = pd.read_csv(ref_dir / 'adamson' / 'adamson_protospacer_sgrna.csv')
adamson_gene_to_protospacer = dict(zip(adamson_df['Gene'].str.strip(), adamson_df['Protospacer'].str.strip()))
print(f'Adamson: {len(adamson_gene_to_protospacer)} gene -> protospacer mappings')
print_list_head(adamson_gene_to_protospacer)

Adamson: 85 gene -> protospacer mappings
{'AARS': 'GAGGGCGGCCTACCTCTCCT', 'AMIGO3/GMPPB': 'GGGGCCAGCAGCCGTCTACC', 'ARHGAP22': 'GGTCCGTCCGGAGCCAGGAG', 'ASCC3': 'GCGCACAGACCCGGCGAGGA', 'ATF6': 'GGGGATCTGAGAATGTACCA', 'ATP5B': 'GAGTCTCCGCAAGGCCCCGG', 'CAD': 'GTAGGAGCCTCGGGCGCGCT', 'CARS': 'GAGCCATGGCAGATTCCTCC', 'CCND3': 'GCGACGTCCGAGCATTCCA', 'CHERP': 'GCGCTGGTGGTCGATCGTG'}


In [7]:
norman_sgrna_df = pd.read_csv(ref_dir / 'norman' / 'norman_sgrna.csv')
norman_target_df = pd.read_csv(ref_dir / 'norman' / 'norman_guideid_ensemble_id_map.csv')
print(f'Norman: {len(norman_sgrna_df)} sgRNA entries')
norman_sgrna_df.head()

Norman: 289 sgRNA entries


,number,gene_A,gene_B,protospacer_sequence_A,protospacer_sequence_B,GBC,Notes
0,1,AHR,NegCtrl0,GAGACGGAATGGAATCCAGA,GTCGCGCCCGCTCCAGGGAC,GAGTCGGACTCCGCCATG,NaN
1,2,ARID1A,NegCtrl0,GCCGCCTGGCAAACCCGGAG,GTCGCGCCCGCTCCAGGGAC,CACAGCATACTAGCGACC,NaN
2,3,ARRDC3,NegCtrl0,GGTACAGTAGGTGTAGAGCT,GTCGCGCCCGCTCCAGGGAC,AAGTTTGAGCGATGCCGT,NaN
3,4,ATL1,NegCtrl0,GAGTGCTCGGGCGGGCCGCT,GTCGCGCCCGCTCCAGGGAC,GCTAAGGGTTTGATGAGG,NaN
4,5,BAK1,NegCtrl0,GCAGGCAGGGCGGCTGTCAG,GTCGCGCCCGCTCCAGGGAC,TCAGACGTGGTGAGATCG,NaN


In [8]:
norman_gene_to_ensg = {}
for _, row in norman_target_df.iterrows():
    if is_valid(row.get('first_target')) and is_valid(row.get('first_id')):
        norman_gene_to_ensg[row['first_target'].strip()] = row['first_id'].strip()
    if is_valid(row.get('second_target')) and is_valid(row.get('second_id')):
        norman_gene_to_ensg[row['second_target'].strip()] = row['second_id'].strip()
print(f'Norman gene -> ENSG: {len(norman_gene_to_ensg)}')
print_list_head(norman_gene_to_ensg)

Norman gene -> ENSG: 105
{'AHR': 'ENSG00000106546', 'FEV': 'ENSG00000163497', 'KLF1': 'ENSG00000105610', 'ARID1A': 'ENSG00000117713', 'ARRDC3': 'ENSG00000113369', 'ATL1': 'ENSG00000198513', 'BAK1': 'ENSG00000030110', 'BCL2L11': 'ENSG00000153094', 'TGFBR2': 'ENSG00000163513', 'BCORL1': 'ENSG00000085185'}


In [9]:
uniprot_fasta = ref_dir / 'uniprot' / 'UP000005640_9606.fasta.gz'
acc_to_seq = {}
with gzip.open(uniprot_fasta, 'rt') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        parts = record.id.split('|')
        if len(parts) >= 2:
            acc_to_seq[parts[1]] = str(record.seq)
print(f'UniProt FASTA: {len(acc_to_seq)} protein sequences')

UniProt FASTA: 20659 protein sequences


## Step 2: Extract Unique Perturbations + Drug Targets

In [10]:
CONTROL_GENES = {'negative_control', 'non-targeting', 'control', 'negctrl', 'negctrl0', 'ctrl', 'nan', '*'}

def get_sequences_from_sgid_ab(sgid_ab):
    if not is_valid(sgid_ab):
        return None, None
    sgid_ab = str(sgid_ab).replace(',', '-').strip()
    parts = sgid_ab.split('|')
    seq_a = sgid_to_seq.get(parts[0].strip())
    seq_b = sgid_to_seq.get(parts[1].strip()) if len(parts) > 1 else None
    return seq_a, seq_b

def get_gene_from_sgid_ab(sgid_ab):
    if not is_valid(sgid_ab):
        return None
    sgid_ab = str(sgid_ab).replace(',', '-').strip()
    parts = sgid_ab.split('|')
    return sgid_to_gene.get(parts[0].strip())

def extract_adamson_gene(pert):
    s = str(pert).strip()
    if '/' in s:
        s = s.split('/')[0].strip()
    if '_p' in s:
        s = s.split('_p')[0].strip()
    return s if s else None

In [11]:
datasets = {
    'k562e_raw': ref_dir / 'raw_k562e' / 'K562_essential_raw_singlecell_01.h5ad',
    'rep1e': ref_dir / 'rep1e' / 'rpe1_raw_singlecell_01.h5ad',
    'k562gw': ref_dir / 'k562gw' / 'K562_gwps_raw_singlecell_01.h5ad',
}

all_crispri_sgids = set()
sgid_ab_to_gene = {}
pert_symbol_to_ensg = {}

for ds_name, ds_path in datasets.items():
    print(f'Loading {ds_name}...')
    adata = sc.read_h5ad(ds_path, backed='r')
    if 'sgID_AB' in adata.obs.columns:
        sgids = set(adata.obs['sgID_AB'].dropna().unique())
        all_crispri_sgids.update(sgids)
        for sgid in sgids:
            if sgid not in sgid_ab_to_gene:
                gene = get_gene_from_sgid_ab(sgid)
                if gene and gene.lower() not in CONTROL_GENES:
                    sgid_ab_to_gene[sgid] = gene
        print(f'Found {len(sgids)} unique sgID_AB')
    
    if 'gene' in adata.obs.columns and 'gene_id' in adata.obs.columns:
        for gene, gene_id in zip(adata.obs['gene'], adata.obs['gene_id']):
            if is_valid(gene) and is_valid(gene_id) and str(gene_id).startswith('ENSG'):
                gene_str = str(gene).strip()
                if gene_str.lower() not in CONTROL_GENES and gene_str not in pert_symbol_to_ensg:
                    pert_symbol_to_ensg[gene_str] = str(gene_id).strip()
        print(f'Extracted {len(pert_symbol_to_ensg)} gene -> ENSG mappings so far')

print(f'Total CRISPRi sgID_AB: {len(all_crispri_sgids)}')
print(f'sgID_AB -> gene mappings: {len(sgid_ab_to_gene)}')
print(f'Perturbation symbol -> ENSG mappings: {len(pert_symbol_to_ensg)}')

Loading k562e_raw...
Found 2273 unique sgID_AB
Extracted 2057 gene -> ENSG mappings so far
Loading rep1e...
Found 2662 unique sgID_AB
Extracted 2392 gene -> ENSG mappings so far
Loading k562gw...
Found 11187 unique sgID_AB
Extracted 9856 gene -> ENSG mappings so far
Total CRISPRi sgID_AB: 11256
sgID_AB -> gene mappings: 10720
Perturbation symbol -> ENSG mappings: 9856


In [12]:
adamson_path = ref_dir / 'adamson' / 'AdamsonWeissman2016_GSM2406681_10X010.h5ad'
adata = sc.read_h5ad(adamson_path, backed='r')
adamson_perts = {p for p in adata.obs['perturbation'].dropna().unique()
                 if is_valid(p) and str(p).strip().lower() not in CONTROL_GENES}
print(f'Adamson perturbations: {len(adamson_perts)}')
list(adamson_perts)[:10]

Adamson perturbations: 113


['TTI1_pDS407',
 'COPB1_pDS065',
 'SEC61A1_pDS031',
 'OST4_pDS353',
 '62(mod)_pBA581',
 'SEL1L_pDS373',
 'UFL1_pDS410',
 'EIF2B2_pDS463',
 'GMPPB_pDS391',
 'YIPF5_pDS001']

In [13]:
norman_path = ref_dir / 'norman' / 'NormanWeissman2019_filtered.h5ad'
adata = sc.read_h5ad(norman_path, backed='r')
norman_non_ctrl_mask = adata.obs['perturbation'].astype(str).str.lower() != 'control'
norman_guide_ids = set(adata.obs.loc[norman_non_ctrl_mask, 'guide_id'].dropna().unique())
print(f'Norman guide_ids (non-control): {len(norman_guide_ids)}')
list(norman_guide_ids)[:10]

Norman guide_ids (non-control): 286


['TSC22D1_NegCtrl0;TSC22D1_NegCtrl0',
 'STIL_NegCtrl0;STIL_NegCtrl0',
 'NegCtrl0_RUNX1T1;NegCtrl0_RUNX1T1',
 'CEBPE_SPI1;CEBPE_SPI1',
 'ZC3HAV1_HOXC13;ZC3HAV1_HOXC13',
 'SAMD1_NegCtrl0;SAMD1_NegCtrl0',
 'UBASH3A_NegCtrl0;UBASH3A_NegCtrl0',
 'TGFBR2_NegCtrl0;TGFBR2_NegCtrl0',
 'NegCtrl0_MAPK1;NegCtrl0_MAPK1',
 'UBASH3B_OSR2;UBASH3B_OSR2']

In [14]:
sciplex_path = ref_dir / 'sciplex' / 'SrivatsanTrapnell2020_sciplex3.h5ad'
adata = sc.read_h5ad(sciplex_path, backed='r')
pert_col = 'product_name' if 'product_name' in adata.obs.columns else 'perturbation'
sciplex_drugs = {d for d in adata.obs[pert_col].dropna().unique()
                 if is_valid(d) and str(d).lower() not in ['control', 'vehicle', 'dmso', 'nan']}
print(f'Sciplex drugs: {len(sciplex_drugs)}')
list(sciplex_drugs)[:10]

Sciplex drugs: 188


['Tofacitinib (CP-690550) Citrate',
 'Pirarubicin',
 'Triamcinolone Acetonide',
 'Costunolide',
 'Streptozotocin (STZ)',
 'Curcumin',
 'Clevudine ',
 'Alvespimycin (17-DMAG) HCl',
 'GSK J1',
 'GSK-LSD1 2HCl']

In [15]:
drug_to_target = {}
if 'target' in adata.obs.columns:
    for drug in sciplex_drugs:
        drug_rows = adata.obs[adata.obs[pert_col] == drug]
        targets = drug_rows['target'].dropna().unique()
        if len(targets) > 0:
            target_name = str(targets[0]).strip()
            if target_name and target_name.lower() not in ['nan', '']:
                drug_to_target[drug] = target_name
print(f'Drug -> target mappings: {len(drug_to_target)}')
print_list_head(drug_to_target)

Drug -> target mappings: 188
{'Tofacitinib (CP-690550) Citrate': 'JAK', 'Pirarubicin': 'Topoisomerase', 'Triamcinolone Acetonide': 'Glucocorticoid Receptor', 'Costunolide': 'Telomerase', 'Streptozotocin (STZ)': 'DNA alkylator', 'Curcumin': 'NF-κB,HDAC,Histone Acetyltransferase,Nrf2', 'Clevudine ': 'DNA/RNA Synthesis', 'Alvespimycin (17-DMAG) HCl': 'HSP (e.g. HSP90)', 'GSK J1': 'Histone Demethylase', 'GSK-LSD1 2HCl': 'Histone Demethylase'}


## Step 3: Build Target Gene List

In [16]:
all_target_genes = set()

for gene in sgid_ab_to_gene.values():
    all_target_genes.add(gene)
list(all_target_genes)[-10:]

['FURIN',
 'ZZZ3',
 'RNASEH2A',
 'ERP29',
 'HSDL2',
 'PRPF4B',
 'ZNF90',
 'EARS2',
 'ASB6',
 'ATP6V1F']

In [17]:
for pert in adamson_perts:
    gene = extract_adamson_gene(pert)
    if gene:
        all_target_genes.add(gene)

In [18]:
for gene in norman_gene_to_ensg.keys():
    all_target_genes.add(gene)

In [19]:
for target_name in drug_to_target.values():
    all_target_genes.add(target_name)

print(f'Total unique target genes: {len(all_target_genes)}')

Total unique target genes: 9985


## Step 4: Map Genes to Protein Sequences (Simplified)

In [20]:
mg = mygene.MyGeneInfo()

gene_identifiers = set()
identifier_to_original = {}

for gene in all_target_genes:
    if gene.startswith('ENSG'):
        gene_identifiers.add(gene)
        identifier_to_original[gene] = gene
    elif gene in pert_symbol_to_ensg:
        ensg = pert_symbol_to_ensg[gene]
        gene_identifiers.add(ensg)
        identifier_to_original[ensg] = gene
    elif gene in norman_gene_to_ensg:
        ensg = norman_gene_to_ensg[gene]
        gene_identifiers.add(ensg)
        identifier_to_original[ensg] = gene
    else:
        gene_identifiers.add(gene)
        identifier_to_original[gene] = gene

print(f'Querying mygene for {len(gene_identifiers)} unique gene identifiers...')
print_list_head(identifier_to_original)
list(gene_identifiers)[:10]

Querying mygene for 9985 unique gene identifiers...
{'ENSG00000223496': 'EXOSC6', 'ENSG00000132256': 'TRIM5', 'ENSG00000117543': 'DPH5', 'ENSG00000181315': 'ZNF322', 'ENSG00000178935': 'ZNF552', 'ENSG00000102921': 'N4BP1', 'ENSG00000105671': 'DDX49', 'ENSG00000186951': 'PPARA', 'ENSG00000079102': 'RUNX1T1', 'ENSG00000205208': 'C4orf46'}


['ENSG00000215717',
 'ENSG00000182359',
 'ENSG00000198131',
 'ENSG00000140443',
 'ENSG00000187961',
 'ENSG00000105127',
 'ENSG00000106299',
 'ENSG00000157014',
 'ENSG00000158411',
 'ENSG00000241343']

In [21]:
all_fields = mg.get_fields(verbose=False)
all_indexed_scopes = ','.join(sorted(k for k, v in all_fields.items() if v.get('index')))
results = mg.querymany(
    list(gene_identifiers),
    scopes=all_indexed_scopes,
    fields='uniprot',
    species='human',
    verbose=False
)

id_to_uniprot = {}
no_uniprot = []
for res in results:
    query = res['query']
    if 'uniprot' in res:
        up = res['uniprot']
        acc = up.get('Swiss-Prot') or up.get('TrEMBL')
        if acc:
            id_to_uniprot[query] = acc[0] if isinstance(acc, list) else acc
        else:
            no_uniprot.append(query)
    else:
        no_uniprot.append(query)

print(f'Found UniProt for {len(id_to_uniprot)} / {len(gene_identifiers)} genes')
print(f'Missing UniProt: {len(set(no_uniprot))}')

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


Found UniProt for 9913 / 9985 genes
Missing UniProt: 122


In [22]:
set(no_uniprot)

{'62(mod)',
 '63(mod)',
 'AHSA2',
 'ATP5E',
 'Androgen Receptor',
 'Aromatase',
 'Aurora Kinase,Bcr-Abl,FLT3',
 'Aurora Kinase,Bcr-Abl,JAK',
 'Aurora Kinase,Bcr-Abl,c-RET,FGFR',
 'Aurora Kinase,CDK',
 'Aurora Kinase,FLT3,VEGFR',
 'Aurora Kinase,VEGFR',
 'Autophagy,ROCK',
 'Autophagy,Sirtuin',
 'BRE',
 'Bcr-Abl',
 'Bcr-Abl,c-Kit,Src',
 'C19orf43',
 'C22orf46',
 'C6orf106',
 'C6orf48',
 'C8orf44',
 'CECR5',
 'CSF-1R,PDGFR,VEGFR',
 'DNA Methyltransferase',
 'DNA alkylator',
 'DSCR3',
 'EGFR',
 'EGFR,HDAC,HER2',
 'EGFR,HER2',
 'ENSG00000112096',
 'ENSG00000125462',
 'ENSG00000130723',
 'ENSG00000153113',
 'ENSG00000158483',
 'ENSG00000167747',
 'ENSG00000167920',
 'ENSG00000168078',
 'ENSG00000175711',
 'ENSG00000177693',
 'ENSG00000177946',
 'ENSG00000184029',
 'ENSG00000188707',
 'ENSG00000189144',
 'ENSG00000189366',
 'ENSG00000196381',
 'ENSG00000197568',
 'ENSG00000197734',
 'ENSG00000203812',
 'ENSG00000205212',
 'ENSG00000215271',
 'ENSG00000230257',
 'ENSG00000255823',
 'ENSG000002

In [23]:
id_to_protein_seq = {}
missing_proteins = []
for id, acc in id_to_uniprot.items():
    if acc in acc_to_seq:
        id_to_protein_seq[id] = acc_to_seq[acc]
    else:
        missing_proteins.append(id)

print(f'Matched {len(id_to_protein_seq)} protein sequences from UniProt FASTA')
print(f'Missing from FASTA: {len(missing_proteins)}')

Matched 9891 protein sequences from UniProt FASTA
Missing from FASTA: 22


In [24]:
genes_to_try_entrez = missing_proteins + no_uniprot
len(set(genes_to_try_entrez))

143

In [25]:
def try_entrez_protein(query_term):
    try:
        search_handle = Entrez.esearch(db='gene', term=query_term, retmax=1)
        search_record = Entrez.read(search_handle)
        if search_record['IdList']:
            ncbi_gene_id = search_record['IdList'][0]
            link_handle = Entrez.elink(dbfrom='gene', db='protein', id=ncbi_gene_id, linkname='gene_protein_refseq')
            link_record = Entrez.read(link_handle)
            if link_record and link_record[0]['LinkSetDb']:
                protein_id = link_record[0]['LinkSetDb'][0]['Link'][0]['Id']
                fetch_handle = Entrez.efetch(db='protein', id=protein_id, rettype='fasta', retmode='text')
                seq_record = SeqIO.read(fetch_handle, 'fasta')
                return str(seq_record.seq)
    except Exception:
        pass
    return None

def fetch_protein_from_ensembl(ensg_id):
    url = f'https://rest.ensembl.org/sequence/id/{ensg_id}?type=protein'
    try:
        r = requests.get(url, headers={'Content-Type': 'text/plain'}, timeout=10)
        if r.ok and len(r.text) > 10:
            return r.text.strip()
    except Exception:
        pass
    return None

def get_first_target(target_str):
    if ',' in target_str and not target_str.startswith('ENSG'):
        return target_str.split(',')[0].strip()
    return target_str

In [26]:
genes_to_try_entrez = missing_proteins + no_uniprot
unmapped_genes = []

if genes_to_try_entrez:
    print(f'Trying Entrez for {len(genes_to_try_entrez)} genes ({len(missing_proteins)} missing from FASTA, {len(no_uniprot)} no UniProt)...')
    entrez_found = 0
    for gene_id in tqdm(genes_to_try_entrez, desc='Entrez lookup'):
        if gene_id in id_to_protein_seq:
            continue
        seq = try_entrez_protein(gene_id)
        if seq:
            id_to_protein_seq[gene_id] = seq
            entrez_found += 1
        else:
            unmapped_genes.append(gene_id)
    print(f'Found {entrez_found} protein sequences via Entrez')

print(f'Protein sequences after Entrez: {len(id_to_protein_seq)}')


Trying Entrez for 186 genes (22 missing from FASTA, 164 no UniProt)...


Entrez lookup: 100%|████████████████████████████████████████████████████████████| 186/186 [01:50<00:00,  1.69it/s]

Found 64 protein sequences via Entrez
Protein sequences after Entrez: 9955


In [27]:
if unmapped_genes:
    print(f'Still unmapped: {len(set(unmapped_genes))}')

Still unmapped: 30


In [28]:
if unmapped_genes:
    print(f'\nTrying fallback sources for {len(unmapped_genes)} unmapped genes...')
    fallback_found = 0
    still_unmapped = []
    
    for gene_id in tqdm(unmapped_genes, desc='Fallback lookup'):
        seq = None
        
        if gene_id.startswith('ENSG'):
            seq = fetch_protein_from_ensembl(gene_id)
            if not seq:
                symbol = identifier_to_original.get(gene_id)
                if symbol and symbol != gene_id:
                    seq = try_entrez_protein(symbol)
        else:
            query = get_first_target(gene_id)
            if query != gene_id:
                seq = try_entrez_protein(query)
        
        if seq:
            id_to_protein_seq[gene_id] = seq
            fallback_found += 1
        else:
            still_unmapped.append(gene_id)
    
    print(f'Found {fallback_found} additional protein sequences via fallback')
    unmapped_genes = still_unmapped

print(f'\nFinal protein sequences: {len(id_to_protein_seq)}')


Trying fallback sources for 30 unmapped genes...


Fallback lookup: 100%|████████████████████████████████████████████████████████████| 30/30 [00:48<00:00,  1.63s/it]

Found 20 additional protein sequences via fallback

Final protein sequences: 9975


In [29]:
if unmapped_genes:
    print(f'Unmapped genes ({len(set(unmapped_genes))}): {set(unmapped_genes)}')

Unmapped genes (10): {'Gal4-4(mod)', 'ENSG00000197734', 'ENSG00000177946', 'ENSG00000125462', 'C22orf46', 'ENSG00000167920', 'ENSG00000188707', 'ENSG00000158483', 'ENSG00000197568', 'ENSG00000189366'}


## Step 5: Map Drugs to SMILES

In [30]:
def get_smiles_robust(raw_name):
    clean_parts = re.split(r'[()\[\]?]', raw_name)
    candidates = [raw_name] + [p.strip() for p in clean_parts if p.strip()]
    candidates = list(dict.fromkeys(candidates))
    for candidate in candidates:
        try:
            compounds = pcp.get_compounds(candidate, 'name')
            if compounds:
                return compounds[0].smiles
        except Exception:
            continue
    return None

In [31]:
drug_to_smiles = {}
missing_drugs = []
for drug in tqdm(sciplex_drugs, desc='Fetching SMILES from PubChem'):
    smiles = get_smiles_robust(drug)
    if smiles:
        drug_to_smiles[drug] = smiles
    else:
        missing_drugs.append(drug)

print(f'Found SMILES for {len(drug_to_smiles)} / {len(sciplex_drugs)} drugs')
if missing_drugs:
    print(f'Missing SMILES for: {missing_drugs}')

Fetching SMILES from PubChem: 100%|█████████████████████████████████████████████| 188/188 [01:01<00:00,  3.08it/s]

Found SMILES for 188 / 188 drugs


## Step 6: Generate DNA Embeddings (Unified dna_to_idx)

In [32]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

nt_model_name = 'InstaDeepAI/NTv3_650M_pre'
nt_tokenizer = AutoTokenizer.from_pretrained(nt_model_name, trust_remote_code=True)
nt_model = AutoModelForMaskedLM.from_pretrained(nt_model_name, trust_remote_code=True)
nt_model.to(device).eval()
print(f'Loaded NucleotideTransformer: {nt_model_name}')

def get_dna_embedding(sequence):
    inputs = nt_tokenizer([sequence], return_tensors='pt', padding='max_length', max_length=128, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    attention_mask = (inputs['input_ids'] != nt_tokenizer.pad_token_id).long()
    with torch.no_grad():
        outputs = nt_model(**inputs, output_hidden_states=True)
    embeddings = outputs.hidden_states[-1]
    mask = attention_mask.unsqueeze(-1).expand(embeddings.size()).float()
    pooled = (embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return pooled.squeeze(0).cpu().numpy()

Loaded NucleotideTransformer: InstaDeepAI/NTv3_650M_pre


In [33]:
dna_vectors = []
dna_to_idx = {}
norman_guide_to_dna_idx = {}

crispri_skipped = []
for sgid_ab in tqdm(all_crispri_sgids, desc='Embedding CRISPRi sgRNAs'):
    seq_a, seq_b = get_sequences_from_sgid_ab(sgid_ab)
    if seq_a is None:
        crispri_skipped.append(sgid_ab)
        continue
    emb_a = get_dna_embedding(seq_a)
    combined = (emb_a + get_dna_embedding(seq_b)) / 2 if seq_b else emb_a
    idx = len(dna_vectors)
    dna_vectors.append(combined)
    dna_to_idx[sgid_ab] = idx

print(f'CRISPRi: {len(dna_vectors)} embeddings (skipped {len(crispri_skipped)})')

Embedding CRISPRi sgRNAs: 100%|█████████████████████████████████████████████| 11256/11256 [33:28<00:00,  5.60it/s]

CRISPRi: 11256 embeddings (skipped 0)


In [34]:
adamson_skipped = []
for pert in tqdm(adamson_perts, desc='Embedding Adamson sgRNAs'):
    gene = extract_adamson_gene(pert)
    if gene is None:
        continue
    protospacer = adamson_gene_to_protospacer.get(gene)
    if protospacer is None or not is_valid(protospacer):
        adamson_skipped.append(pert)
        continue
    emb = get_dna_embedding(protospacer)
    idx = len(dna_vectors)
    dna_vectors.append(emb)
    dna_to_idx[pert] = idx

print(f'Adamson: added {len(adamson_perts) - len(adamson_skipped)} embeddings (skipped {len(adamson_skipped)})')
print(f'Total DNA embeddings: {len(dna_vectors)}')

Embedding Adamson sgRNAs: 100%|█████████████████████████████████████████████████| 113/113 [00:07<00:00, 15.75it/s]

Adamson: added 98 embeddings (skipped 15)
Total DNA embeddings: 11354


In [35]:
for _, row in tqdm(norman_sgrna_df.iterrows(), total=len(norman_sgrna_df), desc='Embedding Norman sgRNAs'):
    gene_a = row['gene_A']
    gene_b = row['gene_B']
    if not is_valid(gene_a):
        continue
    guide_id = f'{gene_a}_{gene_b}' if is_valid(gene_b) and str(gene_b).lower() != 'negctrl0' else gene_a
    seq_a = row['protospacer_sequence_A']
    if not is_valid(seq_a):
        continue
    emb_a = get_dna_embedding(seq_a)
    seq_b = row.get('protospacer_sequence_B')
    if seq_b and is_valid(seq_b) and str(gene_b).lower() != 'negctrl0':
        combined = (emb_a + get_dna_embedding(seq_b)) / 2
    else:
        combined = emb_a
    idx = len(dna_vectors)
    dna_vectors.append(combined)
    dna_to_idx[guide_id] = idx
    norman_guide_to_dna_idx[guide_id] = idx

print(f'Norman: {len(norman_guide_to_dna_idx)} embeddings')
print(f'Total DNA embeddings: {len(dna_vectors)}')

Embedding Norman sgRNAs: 100%|██████████████████████████████████████████████████| 289/289 [00:34<00:00,  8.42it/s]

Norman: 289 embeddings
Total DNA embeddings: 11643


In [36]:
del nt_model, nt_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

dna_embeddings = np.stack(dna_vectors, axis=0).astype(np.float32)
print(f'DNA embeddings shape: {dna_embeddings.shape}')

np.save(seq_banks_dir / 'dna_embeddings.npy', dna_embeddings)
with open(seq_banks_dir / 'dna_to_idx.json', 'w') as f:
    json.dump(dna_to_idx, f)

DNA embeddings shape: (11643, 1536)


## Step 7: Generate Protein Embeddings

In [37]:
from transformers import AutoTokenizer, AutoModel

esm_model_name = 'facebook/esm2_t6_8M_UR50D'
esm_tokenizer = AutoTokenizer.from_pretrained(esm_model_name)
esm_model = AutoModel.from_pretrained(esm_model_name)
esm_model.to(device).eval()
print(f'Loaded ESM-2: {esm_model_name}')

def get_protein_embedding(protein_seq):
    inputs = esm_tokenizer(protein_seq, return_tensors='pt', truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = esm_model(**inputs)
    emb = outputs.last_hidden_state[0].mean(dim=0)
    return emb.cpu().numpy()

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded ESM-2: facebook/esm2_t6_8M_UR50D


In [38]:
protein_vectors = []
id_to_target_idx = {}

for id, protein_seq in tqdm(id_to_protein_seq.items(), desc='Embedding target proteins'):
    emb = get_protein_embedding(protein_seq)
    idx = len(protein_vectors)
    protein_vectors.append(emb)
    id_to_target_idx[id] = idx

print(f'Generated {len(protein_vectors)} protein embeddings')

Embedding target proteins: 100%|██████████████████████████████████████████████| 9975/9975 [07:25<00:00, 22.39it/s]

Generated 9975 protein embeddings


In [39]:
del esm_model, esm_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

protein_embeddings = np.stack(protein_vectors, axis=0).astype(np.float32)
print(f'Protein embeddings shape: {protein_embeddings.shape}')

np.save(target_banks_dir / 'protein_targets.npy', protein_embeddings)

Protein embeddings shape: (9975, 320)


## Step 8: Generate Chemical Embeddings (Unified chemical_to_idx)

In [40]:
chem_model_name = 'Derify/ChemMRL'
chem_tokenizer = AutoTokenizer.from_pretrained(chem_model_name)
chem_model = AutoModel.from_pretrained(chem_model_name)
chem_model.to(device).eval()
print(f'Loaded ChemMRL: {chem_model_name}')

def get_chemical_embedding(smiles):
    inputs = chem_tokenizer(smiles, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = chem_model(**inputs)
    emb = outputs.last_hidden_state[0].mean(dim=0)
    return emb.cpu().numpy()

The repository Derify/ChemMRL contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Derify/ChemMRL .
 You can inspect the repository content at https://hf.co/Derify/ChemMRL.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y
The repository Derify/ChemMRL contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Derify/ChemMRL .
 You can inspect the repository content at https://hf.co/Derify/ChemMRL.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Loaded ChemMRL: Derify/ChemMRL


In [41]:
chemical_vectors = []
chemical_to_idx = {}

for drug, smiles in tqdm(drug_to_smiles.items(), desc='Embedding chemicals'):
    if smiles in chemical_to_idx:
        chemical_to_idx[drug] = chemical_to_idx[smiles]
        continue
    emb = get_chemical_embedding(smiles)
    idx = len(chemical_vectors)
    chemical_vectors.append(emb)
    chemical_to_idx[smiles] = idx
    chemical_to_idx[drug] = idx

print(f'Generated {len(chemical_vectors)} chemical embeddings')
print(f'chemical_to_idx entries: {len(chemical_to_idx)} (includes both SMILES and drug names)')

Embedding chemicals: 100%|██████████████████████████████████████████████████████| 188/188 [00:09<00:00, 19.79it/s]

Generated 188 chemical embeddings
chemical_to_idx entries: 376 (includes both SMILES and drug names)


In [42]:
del chem_model, chem_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if chemical_vectors:
    chemical_embeddings = np.stack(chemical_vectors, axis=0).astype(np.float32)
    print(f'Chemical embeddings shape: {chemical_embeddings.shape}')
    np.save(seq_banks_dir / 'chemical_embeddings.npy', chemical_embeddings)
    with open(seq_banks_dir / 'chemical_to_idx.json', 'w') as f:
        json.dump(chemical_to_idx, f)
else:
    print('No chemical embeddings generated')

Chemical embeddings shape: (188, 1024)


## Step 9: Generate Alignment Pairs

only save aligment pairs since we need seq and target for training. 

In [43]:
def get_split(pert_id, dataset_name):
    splits = dataset_splits.get(dataset_name, {})
    if pert_id in splits.get('train', []):
        return 'train'
    elif pert_id in splits.get('val', []):
        return 'val'
    elif pert_id in splits.get('test', []):
        return 'test'
    return 'train'

def get_target_idx(gene_name):
    idx = id_to_target_idx.get(gene_name, -1)
    if idx >= 0:
        return idx
    ensg = pert_symbol_to_ensg.get(gene_name) or norman_gene_to_ensg.get(gene_name)
    if ensg:
        idx = id_to_target_idx.get(ensg, -1)
    return idx

In [44]:
alignment_pairs = {'train': [], 'val': [], 'test': []}

for sgid_ab, gene in sgid_ab_to_gene.items():
    seq_idx = dna_to_idx.get(sgid_ab, -1)
    target_idx = get_target_idx(gene)
    if seq_idx >= 0 and target_idx >= 0:
        split = get_split(gene, 'k562e_raw')
        alignment_pairs[split].append({'seq_idx': seq_idx, 'target_idx': target_idx, 'modality': 0, 'mode': 0})

print(f'CRISPRi alignment pairs: {sum(len(alignment_pairs[s]) for s in alignment_pairs)}')

CRISPRi alignment pairs: 10710


In [45]:
adamson_pairs_added = 0
for pert in adamson_perts:
    gene = extract_adamson_gene(pert)
    seq_idx = dna_to_idx.get(pert, -1)
    target_idx = get_target_idx(gene) if gene else -1
    if seq_idx >= 0 and target_idx >= 0:
        split = get_split(gene, 'adamson')
        alignment_pairs[split].append({'seq_idx': seq_idx, 'target_idx': target_idx, 'modality': 0, 'mode': 0})
        adamson_pairs_added += 1

print(f'Adamson alignment pairs added: {adamson_pairs_added}')

Adamson alignment pairs added: 98


In [46]:
norman_pairs_added = 0
for guide_id, seq_idx in norman_guide_to_dna_idx.items():
    gene_a = guide_id.split('_')[0]
    target_idx = get_target_idx(gene_a)
    if seq_idx >= 0 and target_idx >= 0:
        split = get_split(guide_id, 'norman')
        if split == 'train' and guide_id not in dataset_splits.get('norman', {}).get('train', []):
            split = get_split(gene_a, 'norman')
        alignment_pairs[split].append({'seq_idx': seq_idx, 'target_idx': target_idx, 'modality': 0, 'mode': 1})
        norman_pairs_added += 1

print(f'Norman alignment pairs added: {norman_pairs_added}')

Norman alignment pairs added: 232


In [47]:
chem_pairs_added = 0
for drug, target_name in drug_to_target.items():
    seq_idx = chemical_to_idx.get(drug, -1)
    target_idx = get_target_idx(target_name)
    if seq_idx >= 0 and target_idx >= 0:
        split = get_split(drug, 'sciplex')
        alignment_pairs[split].append({'seq_idx': seq_idx, 'target_idx': target_idx, 'modality': 2, 'mode': 4})
        chem_pairs_added += 1

print(f'Chemical alignment pairs added: {chem_pairs_added}')

Chemical alignment pairs added: 188


In [48]:
print(f'\nAlignment pairs total: train={len(alignment_pairs["train"])}, val={len(alignment_pairs["val"])}, test={len(alignment_pairs["test"])}')

for split in ['train', 'val', 'test']:
    pairs = alignment_pairs[split]
    if pairs:
        path = pert_embd_dir / split / f'align_{split}.npz'
        np.savez(path,
            seq_idx=np.array([p['seq_idx'] for p in pairs], dtype=np.int32),
            target_idx=np.array([p['target_idx'] for p in pairs], dtype=np.int32),
            modality=np.array([p['modality'] for p in pairs], dtype=np.int8),
            mode=np.array([p['mode'] for p in pairs], dtype=np.int8)
        )
        print(f'Wrote {len(pairs)} alignment pairs to {path}')


Alignment pairs total: train=10797, val=108, test=323
Wrote 10797 alignment pairs to /Users/djemec/data/jepa/v0_6/pert_embd/train/align_train.npz
Wrote 108 alignment pairs to /Users/djemec/data/jepa/v0_6/pert_embd/val/align_val.npz
Wrote 323 alignment pairs to /Users/djemec/data/jepa/v0_6/pert_embd/test/align_test.npz


## Step 10: Build Output Mappings & Save

In [49]:
gene_to_target_idx = {}

for id, idx in id_to_target_idx.items():
    gene_to_target_idx[id] = idx
    if id in identifier_to_original:
        original = identifier_to_original[id]
        if original != id:
            gene_to_target_idx[original] = idx

print_list_head(gene_to_target_idx)

{'ENSG00000215717': 0, 'TMEM167B': 0, 'ENSG00000182359': 1, 'KBTBD3': 1, 'ENSG00000198131': 2, 'ZNF544': 2, 'ENSG00000140443': 3, 'IGF1R': 3, 'ENSG00000187961': 4, 'KLHL17': 4}


In [50]:
for gene, ensg in pert_symbol_to_ensg.items():
    if ensg in id_to_target_idx and gene not in gene_to_target_idx:
        gene_to_target_idx[gene] = id_to_target_idx[ensg]

for gene, ensg in norman_gene_to_ensg.items():
    if ensg in id_to_target_idx and gene not in gene_to_target_idx:
        gene_to_target_idx[gene] = id_to_target_idx[ensg]

print_list_head(gene_to_target_idx)
print(f'gene_to_target_idx: {len(gene_to_target_idx)} mappings')

{'ENSG00000215717': 0, 'TMEM167B': 0, 'ENSG00000182359': 1, 'KBTBD3': 1, 'ENSG00000198131': 2, 'ZNF544': 2, 'ENSG00000140443': 3, 'IGF1R': 3, 'ENSG00000187961': 4, 'KLHL17': 4}
gene_to_target_idx: 19568 mappings


In [51]:
with open(target_banks_dir / 'gene_to_target_idx.json', 'w') as f:
    json.dump(gene_to_target_idx, f)

In [52]:
input_to_id = {}

for sgid_ab in dna_to_idx:
    if '|' in sgid_ab:
        gene = sgid_ab_to_gene.get(sgid_ab)
        if gene:
            idx = dna_to_idx[sgid_ab]
            for ds in ['k562e', 'rep1e', 'k562gw']:
                key = f'{gene}_crispri_{ds}'
                if key not in input_to_id:
                    input_to_id[key] = idx
                    
print_list_head(input_to_id)

{'MAP4K5_crispri_k562e': 0, 'MAP4K5_crispri_rep1e': 0, 'MAP4K5_crispri_k562gw': 0, 'UHRF1_crispri_k562e': 1, 'UHRF1_crispri_rep1e': 1, 'UHRF1_crispri_k562gw': 1, 'CMTM4_crispri_k562e': 2, 'CMTM4_crispri_rep1e': 2, 'CMTM4_crispri_k562gw': 2, 'SLC38A6_crispri_k562e': 3}


In [53]:
for pert in adamson_perts:
    if pert in dna_to_idx:
        gene = extract_adamson_gene(pert)
        if gene:
            key = f'{gene}_crispri_adamson'
            input_to_id[key] = dna_to_idx[pert]

for guide_id, idx in norman_guide_to_dna_idx.items():
    key = f'{guide_id}_crispra_norman'
    input_to_id[key] = idx

for drug, idx in chemical_to_idx.items():
    if drug in drug_to_smiles:
        key = f'{drug}_inhibitor_sciplex'
        input_to_id[key] = idx

print_list_head(input_to_id)
print(f'input_to_id: {len(input_to_id)} entries')

{'MAP4K5_crispri_k562e': 0, 'MAP4K5_crispri_rep1e': 0, 'MAP4K5_crispri_k562gw': 0, 'UHRF1_crispri_k562e': 1, 'UHRF1_crispri_rep1e': 1, 'UHRF1_crispri_k562gw': 1, 'CMTM4_crispri_k562e': 2, 'CMTM4_crispri_rep1e': 2, 'CMTM4_crispri_k562gw': 2, 'SLC38A6_crispri_k562e': 3}
input_to_id: 30178 entries


In [54]:
with open(pert_embd_dir / 'input_to_id.json', 'w') as f:
    json.dump(input_to_id, f)

## Build Skip Sets for Training Shards

Track perturbations that don't have complete embeddings. A perturbation is skipped for training shards only if BOTH sequence AND target embeddings are missing.

In [55]:
crispri_skip_sgids = set()
for sgid_ab in all_crispri_sgids:
    has_dna = sgid_ab in dna_to_idx
    gene = sgid_ab_to_gene.get(sgid_ab)
    has_target = gene and get_target_idx(gene) >= 0
    if not has_dna and not has_target:
        crispri_skip_sgids.add(sgid_ab)

print(f'CRISPRi sgID_ABs: {len(crispri_skip_sgids)}')

CRISPRi sgID_ABs: 0


In [56]:
adamson_skip_perts = set()
for pert in adamson_perts:
    has_dna = pert in dna_to_idx
    gene = extract_adamson_gene(pert)
    has_target = gene and get_target_idx(gene) >= 0
    if not has_dna and not has_target:
        adamson_skip_perts.add(pert)

print(f'Adamson perts: {len(adamson_skip_perts)}')

Adamson perts: 1


In [57]:
norman_skip_guides = set()
for guide_id in norman_guide_ids:
    clean_guide = guide_id.split(';')[0] if ';' in guide_id else guide_id
    has_dna = clean_guide in dna_to_idx or guide_id in dna_to_idx
    parts = clean_guide.split('_')
    gene_a = parts[0] if parts else None
    has_target = gene_a and get_target_idx(gene_a) >= 0
    if not has_dna and not has_target:
        norman_skip_guides.add(guide_id)

print(f'Norman guides: {len(norman_skip_guides)}')

Norman guides: 3


In [58]:
sciplex_skip_drugs = set()
for drug in sciplex_drugs:
    has_chem = drug in chemical_to_idx
    target = drug_to_target.get(drug)
    has_target = target and get_target_idx(target) >= 0
    if not has_chem and not has_target:
        sciplex_skip_drugs.add(drug)

print(f'Sciplex drugs: {len(sciplex_skip_drugs)}')

Sciplex drugs: 0


In [59]:
skip_sets = {
    'crispri_sgids': list(crispri_skip_sgids),
    'adamson_perts': list(adamson_skip_perts),
    'norman_guides': list(norman_skip_guides),
    'sciplex_drugs': list(sciplex_skip_drugs),
}

with open(data_dir / 'skipped_perturbations.json', 'w') as f:
    json.dump(skip_sets, f, indent=2)

print(f'Saved skipped_perturbations.json to {data_dir}')

Saved skipped_perturbations.json to /Users/djemec/data/jepa/v0_6


In [60]:
print('\nOutput files:')
for f in seq_banks_dir.glob('*'):
    print(f'{f.name}')
for f in target_banks_dir.glob('*'):
    print(f'{f.name}')
for split in ['train', 'val', 'test']:
    for f in (pert_embd_dir / split).glob('align_*.npz'):
        print(f'{split}/{f.name}')


Output files:
dna_embeddings.npy
dna_to_idx.json
chemical_embeddings.npy
chemical_to_idx.json
gene_to_target_idx.json
protein_targets.npy
train/align_train.npz
val/align_val.npz
test/align_test.npz


## Summary

In [61]:
print('=== Data Prep Notebook 2 Complete ===')
print(f'\nDNA Embeddings: {dna_embeddings.shape}')
sgid_ab_count = sum(1 for k in dna_to_idx if '|' in k)
adamson_count = sum(1 for k in dna_to_idx if k in adamson_perts)
norman_count = len(norman_guide_to_dna_idx)
print(f'- CRISPRi sgID_AB: {sgid_ab_count}')
print(f'- Adamson: {adamson_count}')
print(f'- Norman: {norman_count}')

print(f'\nProtein Embeddings: {protein_embeddings.shape}')
print(f'- Target genes: {len(id_to_target_idx)}')

if chemical_vectors:
    print(f'\nChemical Embeddings: {chemical_embeddings.shape}')
    print(f'- Unique embeddings: {len(chemical_vectors)}')
    print(f'- Index entries (SMILES + names): {len(chemical_to_idx)}')

print(f'\nAlignment pairs:')
for split in ['train', 'val', 'test']:
    print(f'- {split}: {len(alignment_pairs[split])}')

print(f'\ninput_to_id entries: {len(input_to_id)}')
print(f'gene_to_target_idx entries: {len(gene_to_target_idx)}')

=== Data Prep Notebook 2 Complete ===

DNA Embeddings: (11643, 1536)
- CRISPRi sgID_AB: 11256
- Adamson: 98
- Norman: 289

Protein Embeddings: (9975, 320)
- Target genes: 9975

Chemical Embeddings: (188, 1024)
- Unique embeddings: 188
- Index entries (SMILES + names): 376

Alignment pairs:
- train: 10797
- val: 108
- test: 323

input_to_id entries: 30178
gene_to_target_idx entries: 19568
